# Power Analysis, Effect Size & Sample Size Estimation Template

**Reusable Jupyter Notebook for Statistical Power Analysis using Python**

Based on concepts from *Applied Univariate, Bivariate, and Multivariate Statistics Using Python* (Denis, 2021) — Chapter 5.

This template supports:
- Effect size computation (Cohen's *d*)
- Power and sample size estimation for one- and two-sample *t*-tests
- Power curve visualization
- Sensitivity analysis across effect sizes, α, and power targets

**How to use:** Replace placeholder values with your study parameters. Always report effect sizes alongside *p*-values.


## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.power import TTestIndPower, TTestPower
from scipy import stats

# Optional: better plots
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

print("Libraries loaded successfully.")

## 2. Effect Size — Cohen's *d*

Cohen's *d* quantifies the standardized mean difference:

333d = \left| \frac{\bar{y} - \mu_0}{\sigma} \right|333

Guidelines (Cohen, 1988):
- Small: ~0.2
- Medium: ~0.5
- Large: ~0.8


In [ ]:
def cohens_d(mean1, mean2, sd, paired=False):
    """
    Compute Cohen's d.
    
    Parameters
    ----------
    mean1, mean2 : float
        Sample means (or sample mean and hypothesized mu0 for one-sample)
    sd : float
        Pooled or population standard deviation
    paired : bool
        If True, treat as paired difference (sd of differences)
    """
    return np.abs(mean1 - mean2) / sd

# Example: one-sample (IQ study from chapter)
sample_mean = 130
mu0 = 100
sigma = 15  # typical IQ SD

d = cohens_d(sample_mean, mu0, sigma)
print(f"Cohen's d = {d:.3f}")
print("Interpretation: Large effect" if d >= 0.8 else 
      "Medium effect" if d >= 0.5 else "Small effect")

## 3. One-Sample *t*-Test — Power & Sample Size

Use  from statsmodels.


In [ ]:
# --- USER INPUTS ---
effect_size = 0.5      # expected Cohen's d
alpha = 0.05           # significance level
power = 0.80           # desired power
alternative = 'two-sided'

# Solve for sample size
analysis = TTestPower()
n_required = analysis.solve_power(
    effect_size=effect_size,
    power=power,
    alpha=alpha,
    alternative=alternative
)
print(f"Required sample size (one-sample): {np.ceil(n_required):.0f}")

# Or compute power given n
n_observed = 30
power_achieved = analysis.power(
    effect_size=effect_size,
    nobs=n_observed,
    alpha=alpha,
    alternative=alternative
)
print(f"Power with n={n_observed}: {power_achieved:.3f}")

## 4. Independent Samples *t*-Test — Power & Sample Size

Use . Assumes equal group sizes by default ().


In [ ]:
# --- USER INPUTS ---
effect_size = 0.8      # expected Cohen's d (between groups)
alpha = 0.05
power = 0.80
ratio = 1.0            # n2 / n1

analysis = TTestIndPower()
n_per_group = analysis.solve_power(
    effect_size=effect_size,
    power=power,
    alpha=alpha,
    ratio=ratio,
    alternative='two-sided'
)
print(f"Required n per group: {np.ceil(n_per_group):.0f}")
print(f"Total N ≈ {2 * np.ceil(n_per_group):.0f}")

# Power given sample sizes
n1 = 25
power_achieved = analysis.power(
    effect_size=effect_size,
    nobs1=n1,
    alpha=alpha,
    ratio=ratio
)
print(f"Power with n1={n1} (equal groups): {power_achieved:.3f}")

## 5. Power Curves

Visualize how power changes with sample size for different effect sizes.


In [ ]:
effect_sizes = [0.2, 0.5, 0.8]
sample_sizes = np.arange(5, 150, 5)

analysis = TTestIndPower()

fig, ax = plt.subplots(figsize=(10, 6))
for d in effect_sizes:
    powers = [analysis.power(effect_size=d, nobs1=n, alpha=0.05, ratio=1.0) 
              for n in sample_sizes]
    ax.plot(sample_sizes, powers, label=f"d = {d}", linewidth=2)

ax.axhline(0.80, color="gray", linestyle="--", label="Power = 0.80")
ax.axhline(0.90, color="lightgray", linestyle=":", label="Power = 0.90")
ax.set_xlabel("Sample size per group (n)")
ax.set_ylabel("Statistical Power")
ax.set_title("Power Curves for Independent Samples t-Test (α = 0.05)")
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 6. Sensitivity Analysis

Explore how required sample size changes across a grid of effect sizes and power targets.


In [ ]:
effects = [0.2, 0.3, 0.5, 0.8]
powers = [0.80, 0.90, 0.95]
alpha = 0.05

results = []
analysis = TTestIndPower()

for d in effects:
    for p in powers:
        n = analysis.solve_power(effect_size=d, power=p, alpha=alpha, ratio=1.0)
        results.append({
            "Cohen's d": d,
            "Power": p,
            "n per group": int(np.ceil(n)),
            "Total N": int(2 * np.ceil(n))
        })

df = pd.DataFrame(results)
print(df.to_string(index=False))

## 7. Interpretation Guidelines & Common Pitfalls

### Always remember:
1. **A small *p*-value does not equal scientific importance.** Report effect size.
2. **Power is determined by:** (a) effect size, (b) sample size, (c) α, (d) variance.
3. **You cannot have “too much” power theoretically**, but diminishing returns exist past ~0.90–0.95.
4. **Big data** still requires effect sizes; *p*-values become almost meaningless with huge *n*.
5. **Reducing variance** (screening, blocking, better measurement) increases power more efficiently than just adding subjects in many designs.

### Quick checklist before interpreting results:
- [ ] Effect size reported and interpreted in context
- [ ] Confidence interval or uncertainty measure provided
- [ ] Sample size justified a priori (or power calculated post-hoc with caution)
- [ ] Assumptions of the test checked
- [ ] Multiple comparisons / multiplicity addressed if relevant


## 8. Your Own Analysis

Replace the placeholders below with your study parameters or observed statistics.


In [ ]:
# --- YOUR STUDY PARAMETERS ---
# observed or expected means
mean_treatment = None
mean_control = None
sd_pooled = None

# or observed t / p and n
# t_stat = None
# n1 = None
# n2 = None

# Then compute d, power, etc.
print("Replace placeholders and run your analysis here.")

---
**References**
- Cohen, J. (1988). *Statistical Power Analysis for the Behavioral Sciences* (2nd ed.).
- Denis, D. J. (2021). *Applied Univariate, Bivariate, and Multivariate Statistics Using Python*.
- statsmodels documentation: 
